In [ ]:
# Import libraries and load dataset
import pandas as pd
import numpy as np

column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised','root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class'
]

nslkdd = pd.read_csv('datasets/verified_nslkdd_SMOTE.txt', names=column_names, header=0)
print(f"Total instances: {nslkdd.shape[0]}")
nslkdd.head()

In [ ]:
# Undersample the "neptune" and "normal" subcatagories
sample_neptune = nslkdd[nslkdd['class'] == 'neptune'].sample(n=4689, random_state=42)
sample_normal = nslkdd[nslkdd['class'] == 'normal'].sample(n=10000, random_state=42)
print(f"neptune samples: {len(sample_neptune)}")
print(f"normal samples: {len(sample_normal)}")

In [ ]:
# Create a separate variable called “nslkdd_exclude_normal_neptune” that copies the original dataset but excludes all instances of “normal” and all instances of “neptune”
nslkdd_exclude_normal_neptune = nslkdd[(nslkdd['class'] != 'normal') & (nslkdd['class'] != 'neptune')]
print(f"Excluded dataset instances: {nslkdd_exclude_normal_neptune.shape[0]}")

In [ ]:
# Create a new dataset that concatenates “sample_neptune” + “sample_normal” + “nslkdd_exclude_normal_neptune” 
nslkdd_balanced = pd.concat([sample_neptune, sample_normal, nslkdd_exclude_normal_neptune]).reset_index(drop=True)
print(f"Total instances after balancing: {nslkdd_balanced.shape[0]}")

In [ ]:
# Sanity check
print("Subcategory counts after balancing:")
print(nslkdd_balanced['class'].value_counts())

In [ ]:
# Rename subcategories to major categories

# Dictonary maps all the subcatagories to major catagories
subcategory_to_major = {
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L',
    'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'normal': 'Normal'
}


nslkdd_balanced['class'] = nslkdd_balanced['class'].map(subcategory_to_major)
print("Major category counts:")
print(nslkdd_balanced['class'].value_counts())

In [ ]:
# Export the new dataset: Balanced and subcatagory conversion
nslkdd_balanced.to_csv('datasets/balanced_train.txt', index=False)
print(f"Saved. Total instances: {nslkdd_balanced.shape[0]}")